# ATE end-to-end examples (genriesz)

This notebook demonstrates how to estimate the **Average Treatment Effect (ATE)** with **genriesz**.

We assume the regressor has the form:

- `X = [D, Z...]`, where `D` is a **binary treatment indicator** (`0/1`),
- `Y` is the observed outcome.

We will compute (optionally with cross-fitting):

- **RA**: regression adjustment (plug-in)
- **RW**: Riesz weighting (weighting only)
- **ARW**: augmented Riesz weighting
- **TMLE**: targeted minimum loss estimation (one-step fluctuation)

We also show how to swap the **basis**:
- polynomial features,
- RKHS-style RBF random features,
- nearest-neighbor matching (kNN catchment basis),
- random forest leaf features (optional),
- neural network embeddings (optional).


In [ ]:
import numpy as np

from genriesz import (
    grr_ate,
    SquaredGenerator,
    UKLGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
    RBFRandomFourierBasis,
    KNNCatchmentBasis,
)

rng = np.random.default_rng(0)


## Synthetic data

In [ ]:
# Data-generating process
n = 3000
d_z = 5

Z = rng.normal(size=(n, d_z))

# Treatment assignment: e(Z) = sigmoid(a'Z)
logits = 0.7 * Z[:, 0] - 0.3 * Z[:, 1]
e = 1.0 / (1.0 + np.exp(-logits))
D = rng.binomial(1, e, size=n).astype(float)

# Potential outcomes (constant effect for simplicity)
tau = 1.0
mu0 = 0.5 * Z[:, 0] + 0.25 * Z[:, 1] ** 2
Y0 = mu0 + rng.normal(scale=1.0, size=n)
Y1 = mu0 + tau + rng.normal(scale=1.0, size=n)

Y = D * Y1 + (1.0 - D) * Y0

# Regressor matrix X = [D, Z...]
X = np.column_stack([D, Z])

print("X shape:", X.shape, "Y shape:", Y.shape)


## Example 1: Polynomial basis + treatment interactions

In [ ]:
# Basis on Z, then interact with D (ATE-friendly)
psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

# Generator: Squared loss (always safe / no domain constraints)
gen = SquaredGenerator(C=0.0).as_generator()

res_poly = grr_ate(
    X=X,
    Y=Y,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_poly.summary_text())


## Example 2: RKHS-style basis (RBF random Fourier features)

This approximates an RBF kernel feature map using random Fourier features, then
interacts the features with treatment.


In [ ]:
psi_rff = RBFRandomFourierBasis(
    n_features=500,
    sigma=1.0,
    standardize=True,
    random_state=0,
)
phi_rff = TreatmentInteractionBasis(base_basis=psi_rff)

res_rff = grr_ate(
    X=X,
    Y=Y,
    basis=phi_rff,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_rff.summary_text())


## Example 3: Nearest-neighbor matching (kNN catchment-area basis)

Nearest-neighbor matching can be expressed using a **catchment-area indicator basis**

\[
\phi_j(z) = \mathbf{1}\{c_j \in \mathrm{NN}_k(z)\},
\]

and is shown in the paper to be a special case of squared-loss Riesz regression.

Below we compute a matching-style ATE estimate using the catchment basis directly.
For a fully general GRR workflow, you can also pass the catchment basis as `basis=...`
to `grr_ate`.


In [ ]:
# Matching-style estimate with a kNN catchment basis
# (This mirrors examples/ate_synthetic_nn_matching.py.)

Z_only = X[:, 1:]  # drop D
n_centers = 400

centers = Z_only[:n_centers]
queries = Z_only[n_centers:]

basis_knn = KNNCatchmentBasis(n_neighbors=1).fit(centers)

# For each query point, w_j counts how often center j is selected
Phi = basis_knn(queries)  # (n_queries, n_centers), dense 0/1
w = Phi.sum(axis=0)       # (n_centers,)

# Matching estimator for ATE:
#   theta_hat = mean_{treated} Y - sum_{controls} w_i Y_i / n_treated
D_cent = D[:n_centers]
Y_cent = Y[:n_centers]

treated = (D_cent == 1)
control = (D_cent == 0)

n_treated = treated.sum()
ate_match = float(Y_cent[treated].mean() - (w[control] @ Y_cent[control]) / n_treated)

print("Matching-style ATE estimate:", ate_match)


## Example 4: Random forest leaf basis (optional)

If you have `scikit-learn` installed, you can use a random forest as a **feature map**
via leaf indicators. This keeps GRR convex (linear in parameters) while giving a
flexible nonparametric basis.


In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
    from genriesz.sklearn_basis import RandomForestLeafBasis

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=6,
        random_state=0,
    )

    leaf_basis = RandomForestLeafBasis(rf, include_bias=True).fit(X, Y)

    res_rf = grr_ate(
        X=X,
        Y=Y,
        basis=leaf_basis,
        generator=gen,
        cross_fit=True,
        folds=5,
        random_state=0,
        estimators=("ra", "rw", "arw", "tmle"),
        outcome_models="shared",
        riesz_penalty="l2",
        riesz_lam=1e-3,
        max_iter=300,
        tol=1e-8,
    )

    print(res_rf.summary_text())

except Exception as err:
    print("Skipping random-forest example (scikit-learn not available or failed):")
    print(err)


## Example 5: Neural network embedding basis (optional)

If you have PyTorch installed, you can use a neural network as a **fixed feature map**.
A recommended workflow is:

1. train an embedding network on a separate task,
2. freeze it,
3. use its outputs as features in GRR.

Below we show the mechanics with a small MLP (training is optional).


In [ ]:
try:
    import torch
    from genriesz.torch_basis import MLPEmbeddingNet, TorchEmbeddingBasis

    torch.manual_seed(0)

    net = MLPEmbeddingNet(input_dim=X.shape[1], hidden_dim=64, output_dim=32)
    # (Optional) Train net here on a separate task.
    # For a lightweight demo, we skip training and just use the random initialization.
    nn_basis = TorchEmbeddingBasis(model=net, include_bias=True, device="cpu")

    res_nn = grr_ate(
        X=X,
        Y=Y,
        basis=nn_basis,
        generator=gen,
        cross_fit=True,
        folds=5,
        random_state=0,
        estimators=("ra", "rw", "arw", "tmle"),
        outcome_models="shared",
        riesz_penalty="l2",
        riesz_lam=1e-3,
        max_iter=300,
        tol=1e-8,
    )

    print(res_nn.summary_text())

except Exception as err:
    print("Skipping neural-net example (torch not available or failed):")
    print(err)
